In [1]:
import pandas as pd

In [2]:
# 1. Tablas de referencia (livianas)
airlines = pd.read_csv("../data/raw/airlines.csv")
airports = pd.read_csv("../data/raw/airports.csv")

print("Airlines:", airlines.shape)
print("Airports:", airports.shape)
print("Aeropuertos sin coordenadas:\n", airports[airports["LATITUDE"].isna()])

Airlines: (14, 2)
Airports: (322, 7)
Aeropuertos sin coordenadas:
     IATA_CODE                                            AIRPORT  \
96        ECP    Northwest Florida Beaches International Airport   
234       PBG                  Plattsburgh International Airport   
313       UST  Northeast Florida Regional Airport (St. August...   

              CITY STATE COUNTRY  LATITUDE  LONGITUDE  
96     Panama City    FL     USA       NaN        NaN  
234    Plattsburgh    NY     USA       NaN        NaN  
313  St. Augustine    FL     USA       NaN        NaN  


In [10]:
# 2. Lectura optimizada de flights.csv (solo columnas clave al principio)
cols_clave = [
    "YEAR", "MONTH", "DAY", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "FLIGHT_NUMBER",
    "AIRLINE", "SCHEDULED_DEPARTURE", "DEPARTURE_DELAY", "ARRIVAL_DELAY",
    "CANCELLED", "DIVERTED"
]
flights = pd.read_csv("../data/raw/flights.csv", usecols=cols_clave, low_memory=False)

print("\nTotal filas flights:", len(flights))
print("Nulos por columna:\n", flights.isna().sum())



Total filas flights: 5819079
Nulos por columna:
 YEAR                        0
MONTH                       0
DAY                         0
AIRLINE                     0
FLIGHT_NUMBER               0
ORIGIN_AIRPORT              0
DESTINATION_AIRPORT         0
SCHEDULED_DEPARTURE         0
DEPARTURE_DELAY         86153
ARRIVAL_DELAY          105071
DIVERTED                    0
CANCELLED                   0
dtype: int64


In [11]:
# 3. Chequeo de códigos de aeropuerto raros (ej. numéricos en octubre)
no_iata = flights[~flights["ORIGIN_AIRPORT"].str.isalpha()]
print("\nFilas con ORIGIN_AIRPORT no alfabético:", len(no_iata))
print("Meses afectados:", no_iata["MONTH"].unique())


Filas con ORIGIN_AIRPORT no alfabético: 486165
Meses afectados: [10]


In [12]:
# 4. Top 50 aeropuertos de origen
top_50_origins = (
    flights[flights["ORIGIN_AIRPORT"].str.isalpha()]
    ["ORIGIN_AIRPORT"]
    .value_counts()
    .head(50)
)
print(f"\nCobertura Top 50 aeropuertos: {top_50_origins.sum() / len(flights):.2%}")


Cobertura Top 50 aeropuertos: 74.27%


### Tratamiento

In [13]:
# ==========================================
# 1. Imputación de LATITUDE / LONGITUDE en airports
# ==========================================
# Veamos si existen otros aeropuertos en las mismas ciudades (Panama City, Plattsburgh, St. Augustine)
ciudades_faltantes = airports[airports["LATITUDE"].isna()][["CITY", "STATE"]]
print("Otros aeropuertos en las mismas ciudades:")
print(airports.merge(ciudades_faltantes, on=["CITY", "STATE"]))

# Imputación jerárquica: 1) Misma ciudad/estado -> 2) Promedio del mismo estado
airports["LATITUDE"] = airports["LATITUDE"].fillna(
    airports.groupby(["CITY", "STATE"])["LATITUDE"].transform("mean")
).fillna(
    airports.groupby("STATE")["LATITUDE"].transform("mean")
)

airports["LONGITUDE"] = airports["LONGITUDE"].fillna(
    airports.groupby(["CITY", "STATE"])["LONGITUDE"].transform("mean")
).fillna(
    airports.groupby("STATE")["LONGITUDE"].transform("mean")
)

print("\nNulos restantes en airports después de imputar:")
print(airports[["LATITUDE", "LONGITUDE"]].isna().sum())

Otros aeropuertos en las mismas ciudades:
Empty DataFrame
Columns: [IATA_CODE, AIRPORT, CITY, STATE, COUNTRY, LATITUDE, LONGITUDE]
Index: []

Nulos restantes en airports después de imputar:
LATITUDE     0
LONGITUDE    0
dtype: int64


In [14]:
# ==========================================
# 2. Chequeo de DESTINATION_AIRPORT no alfabético
# ==========================================
dest_no_iata = flights[~flights["DESTINATION_AIRPORT"].str.isalpha()]
print(f"\nFilas con DESTINATION_AIRPORT no alfabético: {len(dest_no_iata)}")
print("Meses afectados en destino:", dest_no_iata["MONTH"].unique())


Filas con DESTINATION_AIRPORT no alfabético: 486165
Meses afectados en destino: [10]


In [15]:
# ==========================================
# 3. Inferencia del mapeo DOT ID (5 dígitos) -> IATA (3 letras)
# ==========================================
# Tomamos vuelos de Septiembre y Noviembre (donde están los códigos IATA reales)
# como referencia por (AIRLINE, FLIGHT_NUMBER, SCHEDULED_DEPARTURE)
ref_flights = flights[flights["MONTH"].isin([9, 11]) & flights["ORIGIN_AIRPORT"].str.isalpha()]
oct_flights = flights[flights["MONTH"] == 10]

# Mapeo de origen usando la moda (el código IATA más frecuente para esa combinación)
route_ref = (
    ref_flights.groupby(["AIRLINE", "FLIGHT_NUMBER", "SCHEDULED_DEPARTURE"])["ORIGIN_AIRPORT"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={"ORIGIN_AIRPORT": "INFERRED_IATA"})
)

oct_mapped = oct_flights.merge(
    route_ref,
    on=["AIRLINE", "FLIGHT_NUMBER", "SCHEDULED_DEPARTURE"],
    how="inner"
)

# Consolidamos el diccionario {DOT_ID: IATA} tomando la moda por cada ID numérico
dot_to_iata = (
    oct_mapped.groupby("ORIGIN_AIRPORT")["INFERRED_IATA"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

print(f"\nTotal de IDs numéricos distintos en octubre: {oct_flights['ORIGIN_AIRPORT'].nunique()}")
print(f"Total de IDs mapeados exitosamente a IATA: {len(dot_to_iata)}")
print("Muestra del mapeo (primeros 10):", dict(list(dot_to_iata.items())[:10]))

# Verificamos si todos los IATA inferidos existen realmente en airports.csv
valid_iatas = set(airports["IATA_CODE"])
invalid_mapped = {k: v for k, v in dot_to_iata.items() if v not in valid_iatas}
print("Mapeos inválidos (no están en airports.csv):", invalid_mapped)


Total de IDs numéricos distintos en octubre: 306
Total de IDs mapeados exitosamente a IATA: 297
Muestra del mapeo (primeros 10): {'10135': 'ABE', '10136': 'ABI', '10140': 'ABQ', '10146': 'ABY', '10154': 'ACK', '10155': 'ACT', '10157': 'ACV', '10158': 'ACY', '10165': 'ADK', '10170': 'ADQ'}
Mapeos inválidos (no están en airports.csv): {}


In [16]:
# 1. Revisar qué IDs faltaron (en origen y en destino)
todos_ids_oct = set(oct_flights["ORIGIN_AIRPORT"]).union(set(oct_flights["DESTINATION_AIRPORT"]))
faltantes = todos_ids_oct - set(dot_to_iata.keys())
print(f"Total IDs únicos (origen + destino) en octubre: {len(todos_ids_oct)}")
print(f"IDs aún sin mapear ({len(faltantes)}):", faltantes)

# 2. Ampliar referencia a TODOS los meses != 10 y relajar llave a (AIRLINE, FLIGHT_NUMBER)
ref_all = flights[(flights["MONTH"] != 10) & flights["ORIGIN_AIRPORT"].str.isalpha()]

fallback_ref = (
    ref_all.groupby(["AIRLINE", "FLIGHT_NUMBER"])["ORIGIN_AIRPORT"]
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={"ORIGIN_AIRPORT": "FALLBACK_IATA"})
)

oct_faltantes = oct_flights[oct_flights["ORIGIN_AIRPORT"].isin(faltantes)].merge(
    fallback_ref, on=["AIRLINE", "FLIGHT_NUMBER"], how="inner"
)

dot_to_iata_extra = (
    oct_faltantes.groupby("ORIGIN_AIRPORT")["FALLBACK_IATA"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

dot_to_iata.update(dot_to_iata_extra)
aun_faltantes = todos_ids_oct - set(dot_to_iata.keys())
print(f"\nIDs mapeados tras fallback: {len(dot_to_iata)} / {len(todos_ids_oct)}")
print("IDs que siguen sin mapear:", aun_faltantes)

# 3. Ver detalle de los vuelos con IDs que sigan sin mapear (si queda alguno)
if aun_faltantes:
    print("\nDetalle de filas sin mapear:")
    print(oct_flights[
        oct_flights["ORIGIN_AIRPORT"].isin(aun_faltantes) |
        oct_flights["DESTINATION_AIRPORT"].isin(aun_faltantes)
    ][["DAY", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "SCHEDULED_DEPARTURE"]].head(15))

Total IDs únicos (origen + destino) en octubre: 307
IDs aún sin mapear (10): {'10666', '14520', '11315', '10581', '15323', '10577', '12129', '10141', '13360', '14098'}

IDs mapeados tras fallback: 306 / 307
IDs que siguen sin mapear: {'10666'}

Detalle de filas sin mapear:
         DAY AIRLINE  FLIGHT_NUMBER ORIGIN_AIRPORT DESTINATION_AIRPORT  \
4871534   31      AS            437          14747               10666   

         SCHEDULED_DEPARTURE  
4871534                 2125  


En el catálogo oficial de códigos del Departamento de Transporte de EE.UU. (DOT Airport Sequence ID), 10666 corresponde a BLI (Bellingham International Airport, WA)

In [17]:
# Verificar que BLI existe en airports.csv
print(airports[airports["IATA_CODE"] == "BLI"])

# Completar el 100% del mapeo (307 / 307)
dot_to_iata["10666"] = "BLI"

# Confirmar que al mapear origen y destino en todo el dataset de 5.8M de filas no queda ningún código inválido
flights["ORIGIN_CLEAN"] = flights["ORIGIN_AIRPORT"].replace(dot_to_iata)
flights["DEST_CLEAN"] = flights["DESTINATION_AIRPORT"].replace(dot_to_iata)

valid_iatas = set(airports["IATA_CODE"])
print("Orígenes fuera de airports.csv:", set(flights["ORIGIN_CLEAN"]) - valid_iatas)
print("Destinos fuera de airports.csv:", set(flights["DEST_CLEAN"]) - valid_iatas)

   IATA_CODE                           AIRPORT        CITY STATE COUNTRY  \
35       BLI  Bellingham International Airport  Bellingham    WA     USA   

    LATITUDE  LONGITUDE  
35  48.79275 -122.53753  
Orígenes fuera de airports.csv: set()
Destinos fuera de airports.csv: set()


### Revisión Final

In [18]:
# ==========================================
# 1. Top 50 real de aeropuertos de origen (con los 12 meses completos)
# ==========================================
top_50_series = flights["ORIGIN_CLEAN"].value_counts().head(50)
top_50_list = top_50_series.index.tolist()

print(f"Cobertura real Top 50 aeropuertos (12 meses): {top_50_series.sum() / len(flights):.2%}")
print("Top 10 aeropuertos:", top_50_list[:10])
print("Puesto 50 (corte):", top_50_series.tail(1).to_dict())

Cobertura real Top 50 aeropuertos (12 meses): 81.07%
Top 10 aeropuertos: ['ATL', 'ORD', 'DFW', 'DEN', 'LAX', 'SFO', 'IAH', 'PHX', 'LAS', 'MSP']
Puesto 50 (corte): {'PIT': 26249}


In [19]:
# ==========================================
# 2. Formato de horas (SCHEDULED_DEPARTURE) y alineación con Open-Meteo
# ==========================================
# Revisar valores extremos (ej. 0, 5, 2359, 2400)
print("\nMínimo y máximo SCHEDULED_DEPARTURE:", flights["SCHEDULED_DEPARTURE"].min(), flights["SCHEDULED_DEPARTURE"].max())
print("Cantidad de vuelos con SCHEDULED_DEPARTURE == 2400:", (flights["SCHEDULED_DEPARTURE"] == 2400).sum())

# Lógica de extracción de hora para cruzar con el clima de Open-Meteo:
# En Open-Meteo, el timestamp '2015-01-01T14:00' tiene:
# - temperature_2m, wind_speed_10m, cloud_cover: instantáneos a las 14:00
# - precipitation, snowfall: acumulado entre las 13:00 y las 14:00 (hora anterior)
# Por lo tanto, si un vuelo sale a las 14:35, la hora truncada es 14:00 (captura el clima instantáneo de las 14:00
# y la lluvia/nieve acumulada en la hora previa inmediata 13:00-14:00).
# Caso borde: 2400 -> corresponde a las 00:00 del día siguiente (o hora 0 si se trunca a nivel día, conviene normalizar 2400 -> 00:00 + 1 día).
dep_hour = (flights["SCHEDULED_DEPARTURE"] // 100) % 24
print("Distribución de horas programadas (0 a 23):")
print(dep_hour.value_counts().sort_index().head(5))


Mínimo y máximo SCHEDULED_DEPARTURE: 1 2359
Cantidad de vuelos con SCHEDULED_DEPARTURE == 2400: 0
Distribución de horas programadas (0 a 23):
SCHEDULED_DEPARTURE
0    14664
1     5159
2     1414
3      778
4      531
Name: count, dtype: int64


In [20]:
# ==========================================
# 3. Población Gold B, balance de clases y cortes temporales
# ==========================================
# Población Gold B: vuelos completados (ni cancelados ni desviados) y con clima en origen (Top 50)
gold_b_pop = flights[
    (flights["CANCELLED"] == 0) &
    (flights["DIVERTED"] == 0) &
    (flights["ORIGIN_CLEAN"].isin(top_50_list))
].copy()

gold_b_pop["TARGET_DELAY_15"] = (gold_b_pop["ARRIVAL_DELAY"] >= 15).astype(int)

print(f"\nFilas totales en F1: {len(flights):,}")
print(f"Filas elegibles para Gold B (completados + Top 50): {len(gold_b_pop):,} ({len(gold_b_pop)/len(flights):.2%})")
print(f"Tasa global de retraso >= 15 min (desbalance de clases): {gold_b_pop['TARGET_DELAY_15'].mean():.2%}")

# Distribución mensual para definir los cortes Train / Validation / Test
monthly_stats = gold_b_pop.groupby("MONTH").agg(
    vuelos=("TARGET_DELAY_15", "count"),
    tasa_retraso=("TARGET_DELAY_15", "mean")
)
monthly_stats["pct_acumulado"] = monthly_stats["vuelos"].cumsum() / len(gold_b_pop)
print("\nDistribución mensual y acumulado para corte temporal:")
print(monthly_stats)


Filas totales en F1: 5,819,079
Filas elegibles para Gold B (completados + Top 50): 4,637,422 (79.69%)
Tasa global de retraso >= 15 min (desbalance de clases): 19.07%

Distribución mensual y acumulado para corte temporal:
       vuelos  tasa_retraso  pct_acumulado
MONTH                                     
1      369127      0.211710       0.079597
2      329563      0.236346       0.150663
3      397844      0.198233       0.236453
4      388937      0.175913       0.320323
5      398084      0.188397       0.406164
6      399975      0.243250       0.492414
7      416908      0.219077       0.582314
8      408636      0.193179       0.670432
9      375333      0.132733       0.751367
10     393380      0.126244       0.836195
11     376550      0.155034       0.917393
12     383085      0.210849       1.000000
